In [1]:
import pandas as pd
import numpy as np
import ast
import re
import json
from collections import Counter

In [2]:
df_career = pd.read_csv("career_dataset.csv")

print("Dataset loaded successfully!")
print("Rows:", df_career.shape[0])
print("Columns:", df_career.shape[1])

display(df_career.head())

Dataset loaded successfully!
Rows: 970
Columns: 6


,ID_num,job_title,Short_description,Skills_required,Industry,Pay_grade
0,1,Software Engineer,Develop and maintain web applications using mo...,Problem Solving Logical Reasoning Attention to...,Technology,High paying
1,2,Data Scientist,Analyze large datasets to extract business ins...,Analytical Thinking Pattern Recognition Mathem...,Technology,High paying
2,3,Marketing Manager,Lead marketing campaigns and brand strategy de...,Creative Thinking Strategic Planning Communica...,Marketing,Average paying
3,4,UX Designer,Design user-friendly interfaces and improve us...,Creative Problem Solving Empathy Research Skil...,Technology,Average paying
4,5,Financial Analyst,Analyze financial data and prepare reports for...,Analytical Thinking Attention to Detail Mathem...,Finance,Average paying


In [3]:
print("Shape:", df_career.shape)

print("\nColumns:")
print(df_career.columns.tolist())

print("\nDataset information:")
df_career.info()


Shape: (970, 6)

Columns:
['ID_num', 'job_title', 'Short_description', 'Skills_required', 'Industry', 'Pay_grade']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970 entries, 0 to 969
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ID_num             970 non-null    int64 
 1   job_title          970 non-null    object
 2   Short_description  970 non-null    object
 3   Skills_required    970 non-null    object
 4   Industry           970 non-null    object
 5   Pay_grade          970 non-null    object
dtypes: int64(1), object(5)
memory usage: 45.6+ KB


In [4]:
print("Missing values:")
display(df_career.isnull().sum())

Missing values:


ID_num               0
job_title            0
Short_description    0
Skills_required      0
Industry             0
Pay_grade            0
dtype: int64

In [5]:
print("Duplicate rows:",
      df_career.duplicated().sum())

print("Duplicate IDs:",
      df_career["ID_num"].duplicated().sum())


Duplicate rows: 0
Duplicate IDs: 0


In [6]:
df_career.columns = (
    df_career.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df_career.columns.tolist())

['id_num', 'job_title', 'short_description', 'skills_required', 'industry', 'pay_grade']


In [7]:
text_columns = [
    "job_title",
    "short_description",
    "skills_required",
    "industry",
    "pay_grade"
]

for col in text_columns:
    df_career[col] = (
        df_career[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text cleaning completed.")

Text cleaning completed.


In [8]:
df_career["id_num"] = pd.to_numeric(
    df_career["id_num"],
    errors="coerce"
)

print(df_career["id_num"].describe())

count    970.000000
mean     485.500000
std      280.159181
min        1.000000
25%      243.250000
50%      485.500000
75%      727.750000
max      970.000000
Name: id_num, dtype: float64


In [9]:
# Cell 9: Parse Required Skills

def parse_skills(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    # Try Python-list format
    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                str(skill).strip()
                for skill in parsed
                if str(skill).strip()
            ]

    except (ValueError, SyntaxError):
        pass

    # Otherwise split common delimiters
    skills = re.split(r",|;|\||/", value)

    return [
        skill.strip()
        for skill in skills
        if skill.strip()
    ]


df_career["skills_list"] = (
    df_career["skills_required"]
    .apply(parse_skills)
)

display(
    df_career[
        ["job_title", "skills_required", "skills_list"]
    ].head()
)


,job_title,skills_required,skills_list
0,Software Engineer,Problem Solving Logical Reasoning Attention to...,[Problem Solving Logical Reasoning Attention t...
1,Data Scientist,Analytical Thinking Pattern Recognition Mathem...,[Analytical Thinking Pattern Recognition Mathe...
2,Marketing Manager,Creative Thinking Strategic Planning Communica...,[Creative Thinking Strategic Planning Communic...
3,UX Designer,Creative Problem Solving Empathy Research Skil...,[Creative Problem Solving Empathy Research Ski...
4,Financial Analyst,Analytical Thinking Attention to Detail Mathem...,[Analytical Thinking Attention to Detail Mathe...


In [10]:
# Normalize Skills

def normalize_skill(skill):

    skill = str(skill).strip().lower()

    skill = re.sub(
        r"\s+",
        " ",
        skill
    )

    return skill


df_career["normalized_skills"] = (
    df_career["skills_list"]
    .apply(
        lambda skills: [
            normalize_skill(skill)
            for skill in skills
        ]
    )
)

print("Skill normalization completed.")


Skill normalization completed.


In [11]:
##  Career Distribution
print("Number of unique career/job titles:",
      df_career["job_title"].nunique())

print("\nMost common career titles:")

display(
    df_career["job_title"]
    .value_counts()
    .head(20)
)

Number of unique career/job titles: 896

Most common career titles:


job_title
Remote Digital Product Experience Lead    4
Online Digital Brand Experience Lead      3
Remote Digital Product Content Analyst    3
Delivery Driver                           2
Chief Marketing Officer                   2
Security Guard                            2
Training Coordinator                      2
Physical Therapist                        2
Graphic Designer                          2
Biomedical Engineer                       2
Receptionist                              2
Janitor                                   2
Cashier                                   2
Customer Service Representative           2
Compensation Analyst                      2
Retail Sales Associate                    2
Occupational Therapist                    2
Social Media Manager                      2
Library Assistant                         2
Logistics Coordinator                     2
Name: count, dtype: int64

In [12]:
print("Industry distribution:")

display(
    df_career["industry"]
    .value_counts()
)

Industry distribution:


industry
Technology         197
Marketing           94
Healthcare          56
Education           36
Human Resources     35
                  ... 
Energy Research      1
Energy Services      1
Food Services        1
Arts & Sciences      1
Energy               1
Name: count, Length: 105, dtype: int64

In [13]:
print("Pay grade distribution:")

display(
    df_career["pay_grade"]
    .value_counts()
)


Pay grade distribution:


pay_grade
Average paying    545
High paying       285
Low paying        140
Name: count, dtype: int64

In [14]:
skill_counter = Counter()

for skills in df_career["normalized_skills"]:

    for skill in set(skills):

        skill_counter[skill] += 1


skill_frequency = pd.DataFrame(
    skill_counter.items(),
    columns=[
        "Skill",
        "Career_Count"
    ]
)

skill_frequency = (
    skill_frequency
    .sort_values(
        by="Career_Count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Most common career skills:")

display(
    skill_frequency.head(20)
)



Most common career skills:


,Skill,Career_Count
0,communication,304
1,technology,225
2,analytics,118
3,reporting,88
4,attention to detail,53
5,leadership,49
6,customer service,46
7,organization,46
8,branding,31
9,problem solving,20


In [15]:
# Skill Demand Percentage


total_careers = len(df_career)

skill_frequency["Demand_Percentage"] = (
    skill_frequency["Career_Count"]
    / total_careers
) * 100

display(
    skill_frequency.head(20)
)

,Skill,Career_Count,Demand_Percentage
0,communication,304,31.340206
1,technology,225,23.195876
2,analytics,118,12.164948
3,reporting,88,9.072165
4,attention to detail,53,5.463918
5,leadership,49,5.051546
6,customer service,46,4.742268
7,organization,46,4.742268
8,branding,31,3.195876
9,problem solving,20,2.061856


In [16]:
# Career Skill Profiles


career_skill_profiles = (
    df_career[
        [
            "job_title",
            "industry",
            "pay_grade",
            "normalized_skills"
        ]
    ]
    .copy()
)

display(
    career_skill_profiles.head(10)
)


,job_title,industry,pay_grade,normalized_skills
0,Software Engineer,Technology,High paying,[problem solving logical reasoning attention t...
1,Data Scientist,Technology,High paying,[analytical thinking pattern recognition mathe...
2,Marketing Manager,Marketing,Average paying,[creative thinking strategic planning communic...
3,UX Designer,Technology,Average paying,[creative problem solving empathy research ski...
4,Financial Analyst,Finance,Average paying,[analytical thinking attention to detail mathe...
5,Project Manager,Business Services,Average paying,[leadership organization time management commu...
6,Sales Representative,Sales,Average paying,[persuasion communication relationship buildin...
7,HR Specialist,Human Resources,Average paying,[interpersonal skills communication conflict r...
8,Graphic Designer,Creative Services,Average paying,[creativity visual communication color theory ...
9,Business Analyst,Business Services,Average paying,[analytical thinking process improvement probl...


In [17]:
# Skill Gap Function


def calculate_skill_gap(
    student_skills,
    career_skills
):

    student_skills = set(
        normalize_skill(skill)
        for skill in student_skills
    )

    career_skills = set(
        normalize_skill(skill)
        for skill in career_skills
    )

    missing_skills = career_skills - student_skills

    if len(career_skills) == 0:
        gap_percentage = 0
    else:
        gap_percentage = (
            len(missing_skills)
            / len(career_skills)
        ) * 100

    return (
        sorted(missing_skills),
        gap_percentage
    )



In [18]:
def skill_gap_priority(gap_percentage):

    if gap_percentage >= 75:
        return "Critical"

    elif gap_percentage >= 50:
        return "High"

    elif gap_percentage >= 25:
        return "Medium"

    else:
        return "Low"

In [19]:
# This is ONLY an example to demonstrate
# how the documented skill-gap mechanism works.

example_student_skills = [
    "Python",
    "SQL",
    "Machine Learning",
    "Pandas",
    "NumPy"
]

print("Example student skills:")
print(example_student_skills)


Example student skills:
['Python', 'SQL', 'Machine Learning', 'Pandas', 'NumPy']


In [20]:

gap_results = []

for _, row in df_career.iterrows():

    missing_skills, gap_percentage = (
        calculate_skill_gap(
            example_student_skills,
            row["normalized_skills"]
        )
    )

    priority = skill_gap_priority(
        gap_percentage
    )

    gap_results.append({
        "job_title": row["job_title"],
        "missing_skills": missing_skills,
        "skill_gap_percentage": gap_percentage,
        "skill_gap_priority": priority
    })


skill_gap_df = pd.DataFrame(
    gap_results
)

display(
    skill_gap_df.head(20)
)

,job_title,missing_skills,skill_gap_percentage,skill_gap_priority
0,Software Engineer,[problem solving logical reasoning attention t...,100.0,Critical
1,Data Scientist,[analytical thinking pattern recognition mathe...,100.0,Critical
2,Marketing Manager,[creative thinking strategic planning communic...,100.0,Critical
3,UX Designer,[creative problem solving empathy research ski...,100.0,Critical
4,Financial Analyst,[analytical thinking attention to detail mathe...,100.0,Critical
5,Project Manager,[leadership organization time management commu...,100.0,Critical
6,Sales Representative,[persuasion communication relationship buildin...,100.0,Critical
7,HR Specialist,[interpersonal skills communication conflict r...,100.0,Critical
8,Graphic Designer,[creativity visual communication color theory ...,100.0,Critical
9,Business Analyst,[analytical thinking process improvement probl...,100.0,Critical


In [21]:
career_analysis = df_career[
    [
        "id_num",
        "job_title",
        "short_description",
        "industry",
        "pay_grade",
        "normalized_skills"
    ]
].copy()

career_analysis = career_analysis.merge(
    skill_gap_df,
    on="job_title",
    how="left"
)

display(
    career_analysis.head(10)
)


,id_num,job_title,short_description,industry,pay_grade,normalized_skills,missing_skills,skill_gap_percentage,skill_gap_priority
0,1,Software Engineer,Develop and maintain web applications using mo...,Technology,High paying,[problem solving logical reasoning attention t...,[problem solving logical reasoning attention t...,100.0,Critical
1,2,Data Scientist,Analyze large datasets to extract business ins...,Technology,High paying,[analytical thinking pattern recognition mathe...,[analytical thinking pattern recognition mathe...,100.0,Critical
2,3,Marketing Manager,Lead marketing campaigns and brand strategy de...,Marketing,Average paying,[creative thinking strategic planning communic...,[creative thinking strategic planning communic...,100.0,Critical
3,4,UX Designer,Design user-friendly interfaces and improve us...,Technology,Average paying,[creative problem solving empathy research ski...,[creative problem solving empathy research ski...,100.0,Critical
4,5,Financial Analyst,Analyze financial data and prepare reports for...,Finance,Average paying,[analytical thinking attention to detail mathe...,[analytical thinking attention to detail mathe...,100.0,Critical
5,6,Project Manager,Coordinate cross-functional teams to deliver p...,Business Services,Average paying,[leadership organization time management commu...,[leadership organization time management commu...,100.0,Critical
6,7,Sales Representative,Generate leads and close deals with potential ...,Sales,Average paying,[persuasion communication relationship buildin...,[persuasion communication relationship buildin...,100.0,Critical
7,8,HR Specialist,Manage recruitment processes and employee rela...,Human Resources,Average paying,[interpersonal skills communication conflict r...,[interpersonal skills communication conflict r...,100.0,Critical
8,9,Graphic Designer,Create visual content for marketing materials ...,Creative Services,Average paying,[creativity visual communication color theory ...,[creativity visual communication color theory ...,100.0,Critical
9,9,Graphic Designer,Create visual content for marketing materials ...,Creative Services,Average paying,[creativity visual communication color theory ...,"[adobe creative suite, brand guidelines, creat...",100.0,Critical


In [22]:
print("""
The documented MCDA Career Score requires:

Skill       = 30%
Interest    = 20%
Academic    = 15%
Market      = 15%
Alumni      = 10%
Location    = 10%

The current career_dataset.csv does not contain
these six numerical inputs.

Therefore, no artificial scores are generated here.
""")



The documented MCDA Career Score requires:

Skill       = 30%
Interest    = 20%
Academic    = 15%
Market      = 15%
Alumni      = 10%
Location    = 10%

The current career_dataset.csv does not contain
these six numerical inputs.

Therefore, no artificial scores are generated here.



In [23]:
required_mcda_components = [
    "skill_score",
    "interest_score",
    "academic_score",
    "market_score",
    "alumni_score",
    "location_score"
]

available_columns = set(
    df_career.columns
)

missing_mcda_components = [
    col
    for col in required_mcda_components
    if col not in available_columns
]

print("MCDA components missing from raw dataset:")
print(missing_mcda_components)

MCDA components missing from raw dataset:
['skill_score', 'interest_score', 'academic_score', 'market_score', 'alumni_score', 'location_score']


In [25]:
career_records = []

for _, row in career_analysis.iterrows():

    record = {
        "id": int(row["id_num"])
        if not pd.isna(row["id_num"])
        else None,

        "job_title": row["job_title"],

        "description": row[
            "short_description"
        ],

        "industry": row["industry"],

        "pay_grade": row["pay_grade"],

        "required_skills": row[
            "normalized_skills"
        ],

        "missing_skills": row[
            "missing_skills"
        ],

        "skill_gap_percentage": float(
            row["skill_gap_percentage"]
        ),

        "skill_gap_priority": row[
            "skill_gap_priority"
        ]
    }

    career_records.append(record)

In [26]:
with open(
    "careers.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        career_records,
        f,
        indent=2,
        ensure_ascii=False
    )

print("careers.json saved successfully!")


careers.json saved successfully!


In [27]:
career_analysis.to_csv(
    "careers_processed.csv",
    index=False
)

print(
    "careers_processed.csv saved successfully!"
)

careers_processed.csv saved successfully!


In [28]:

print("====================================")
print("CAREER PATHWAYS FINAL VALIDATION")
print("====================================")

print(
    "Total career records:",
    len(career_analysis)
)

print(
    "Unique career titles:",
    career_analysis[
        "job_title"
    ].nunique()
)

print(
    "Missing job titles:",
    career_analysis[
        "job_title"
    ].isna().sum()
)

print("\nSkill-gap priority distribution:")

display(
    career_analysis[
        "skill_gap_priority"
    ].value_counts()
)

print("\nFiles created:")
print("1. careers.json")
print("2. careers_processed.csv")

print("\nCareer Pathways preprocessing completed!")

CAREER PATHWAYS FINAL VALIDATION
Total career records: 1128
Unique career titles: 896
Missing job titles: 0

Skill-gap priority distribution:


skill_gap_priority
Critical    1126
High           2
Name: count, dtype: int64


Files created:
1. careers.json
2. careers_processed.csv

Career Pathways preprocessing completed!
